In [ ]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import csv
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import open_clip
from PIL import Image

In [ ]:
IS_COLAB = True

DATA_DIR = Path('/content/drive/MyDrive/Projects/SnuAiChallenge/Data') if IS_COLAB else Path('../data')
DATASET_DIR = DATA_DIR / 'snuaichallenge_data'

CKPT_PATH = DATA_DIR / 'ckpt'
REPORT_PATH = DATA_DIR / 'Reports'

In [ ]:
if not DATASET_DIR.exists():
    import kagglehub

    kagglehub.login()
    kagglehub.competition_download("snuaichallenge", output_dir=DATA_DIR)

# Pre-processing

In [ ]:
_, clip_train_image_preprocess, clip_test_image_preprocess = \
    open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [ ]:
from typing import Literal


class SnuAiChallengeDataset(Dataset):
    """
    SNU AI Challenge dataset.

    Attributes
    ----------
    metadata : list[tuple]
        A list of rows with ["Id", "Input_1", "Input_2", "Input_3", "Input_4", "Sentence"] information.
    """
    default_transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    def __init__(self, path: Path, variant: Literal['train', 'test'], transform=None):
        self.path = path
        self.variant = variant
        self.transform = transform if transform else default_transform

        self._load_metadata()
        self.cls2idx = {
            (1, 2, 3, 4): 0,
            (1, 2, 4, 3): 1,
            (1, 3, 2, 4): 2,
            (1, 3, 4, 2): 3,
            (1, 4, 2, 3): 4,
            (1, 4, 3, 2): 5,
            (2, 1, 3, 4): 6,
            (2, 1, 4, 3): 7,
            (2, 3, 1, 4): 8,
            (2, 3, 4, 1): 9,
            (2, 4, 1, 3): 10,
            (2, 4, 3, 1): 11,
            (3, 1, 2, 4): 12,
            (3, 1, 4, 2): 13,
            (3, 2, 1, 4): 14,
            (3, 2, 4, 1): 15,
            (3, 4, 1, 2): 16,
            (3, 4, 2, 1): 17,
            (4, 1, 2, 3): 18,
            (4, 1, 3, 2): 19,
            (4, 2, 1, 3): 20,
            (4, 2, 3, 1): 21,
            (4, 3, 1, 2): 22,
            (4, 3, 2, 1): 23,
        }

    def _load_metadata(self):
        self.metadata = []

        with open(self.path / f'{self.variant}.csv') as f:
            csv_reader = csv.reader(f)
            next(csv_reader)  # skip header

            for row in csv_reader:
                self.metadata.append(tuple(row))

    def __getitem__(self, idx: int):
        image_id, image_1, image_2, image_3, image_4, sentence, *etc = self.metadata[idx]

        images = [
            Image.open(self.path / self.variant / image_id / image_filename).convert('RGB')
            for image_filename in [image_1, image_2, image_3, image_4]
        ]
        images = [self.transform(image) for image in images]
        images = torch.stack(images)

        if self.variant == 'train':
            answer, _is_no_ordering = etc

            label = self.cls2idx[tuple(eval(answer))]
            label = torch.tensor(label).long()

            return images, sentence, label

        return images, sentence

    def __len__(self):
        return len(self.metadata)

In [ ]:
train_dataset = SnuAiChallengeDataset(DATASET_DIR, 'train', transform=clip_train_image_preprocess)
test_dataset = SnuAiChallengeDataset(DATASET_DIR, 'test', transform=clip_test_image_preprocess)

# Model

For this task, people usually use a **multimodal ranking or classification model**: one encoder for the text, one encoder for each frame, then a scorer that predicts the best ordering of the 4 frames. The most practical designs are transformer-based, and the best balance of accuracy vs. memory is usually a dual-encoder or a lightly fused encoder rather than a fully joint cross-attention stack. [arxiv](https://arxiv.org/html/2408.12475v1)

## Common architectures

- **Dual-encoder + pairwise scoring.** Encode the text once, encode each frame independently, and score each text–frame pair; then solve the 4-frame ordering with permutation scoring or pairwise order preferences. This is memory-friendly because frame embeddings can be cached or computed independently, and the paper on frame identification explicitly highlights dual encoders as efficient for scoring one input against many candidates. [arxiv](https://arxiv.org/html/2408.12475v1)

- **Siamese / shared-weight encoder.** Use the same encoder backbone for all frames and the text, with a small pooling head and an ordering head. This is usually lighter than two separate towers, and works well when the visual and text sides can share a representation space, but it is less expressive than a fused cross-attention model. [arxiv](https://arxiv.org/html/2408.12475v1)

- **Late-fusion transformer.** Encode text and each frame separately, then concatenate the 5 embeddings and run a small transformer or MLP over them to predict the permutation. This is a good middle ground: more context than pure pairwise scoring, but far less memory than full frame-by-frame cross-attention over all tokens. [honors.cs.umd](http://honors.cs.umd.edu/reports/Vertex_Reordering_for_Cache_Coherency.pdf)

- **Cross-encoder / fused encoder.** Put text and all 4 frames into one transformer and let self-attention model all interactions directly. This is often the strongest for ranking, but it is the most memory-hungry because attention cost grows with the total token count, and the cited frame-identification work notes that fused encoders are less efficient at inference than dual-encoder designs. [arxiv](https://arxiv.org/html/2408.12475v1)

- **Coarse-to-fine two-stage model.** First narrow candidate orders or pairwise relations with a cheap encoder, then refine only the most likely permutations with a heavier interaction module. This is a strong compromise when you want better performance without paying full cross-encoder cost on every example. [arxiv](https://arxiv.org/html/2408.12475v1)

## Best practical choices

For only 4 frames, I would usually start with one of these:
1. **Dual-encoder + permutation head** for the best memory efficiency.
2. **Late-fusion transformer** for the best balance of accuracy and cost.
3. **Cross-encoder** only if accuracy matters more than memory and latency. [arxiv](https://arxiv.org/html/2408.12475v1)

A useful intuition is that the task can be viewed as predicting one of \(4! = 24\) permutations, so you do not need a huge generative model; a compact encoder plus an order classifier is often enough.

## Recommended stack

If you want a strong but lean architecture, a good default is:
- text encoder: small or base Transformer,
- frame encoder: shared-weight visual encoder for each frame,
- fusion: 2–4 layer transformer or MLP over the 5 embeddings,
- head: 24-way permutation classifier or pairwise-ordering head with decoding.

If you want the simplest high-performance baseline, use a **cross-encoder with a small backbone**; if you want the best efficiency, use a **dual encoder with cached frame features**. [arxiv](https://arxiv.org/html/2408.12475v1)

## Training setup

The most common objective is either:
- **24-class classification** over all permutations,
- **pairwise ordering loss** over frame pairs,
- or **ranking loss** on the correct permutation versus sampled wrong ones.

Pairwise training is usually easier to optimize and lighter in memory, while direct permutation classification is simpler when the dataset is clean and not too large.

## What I’d avoid

I would avoid a full autoregressive sequence generator unless you specifically need textual explanation or variable-length outputs. For your fixed 4-number output, generation is usually less efficient and more brittle than classification or ranking.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42

hps = {
    'experiment_number': 1,
    'n_epochs': 1,
    'batch_size': 32,
    'lr': 1e-3,
}

In [ ]:
class O4FFusionTransformer(nn.Module):
    def __init__(self, text_encoder, frames_encoder, tokenizer, frames_dim=512, text_dim=512, hidden_dim=64,  num_classes=24, num_heads=8, num_fusion_layers=2, dropout=0.1):
        super().__init__()

        self.text_encoder = text_encoder
        self.frames_encoder = frames_encoder
        self.tokenizer = tokenizer

        self.frames_proj = nn.Linear(frames_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)

        transformer_encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim * 6, nhead=num_heads, dim_feedforward=hidden_dim * 4, dropout=dropout, batch_first=True, activation='gelu')
        self.fusion = nn.TransformerEncoder(transformer_encoder_layer, num_layers=num_fusion_layers)

        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim))
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 6),
            nn.Linear(hidden_dim * 6, num_classes),
        )

    def forward(self, frames, sentences):
        """
        sentences: (B, T)
        frame_feats: (B, 4, frames_dim)
        """
        input_ids = self.tokenizer(sentences).to(frames.device)
        text_feats = self.text_encoder(input_ids)
        text_feats /= text_feats.norm(dim=-1, keepdim=True)
        text_feats = self.text_proj(text_feats)

        frame_feats = self.frames_encoder(frames.view(frames.shape[0] * frames.shape[1], *frames.shape[2:]))
        frame_feats /= frame_feats.norm(dim=-1, keepdim=True)
        frame_feats = self.frames_proj(frame_feats)
        frame_feats = frame_feats.view(frames.shape[0], frames.shape[1], frame_feats.shape[-1])

        cls = self.cls_token.expand(input_ids.size(0), -1, -1).squeeze()
        x = torch.cat([
            cls,
            text_feats,
            frame_feats[:, 0, :],
            frame_feats[:, 1, :],
            frame_feats[:, 2, :],
            frame_feats[:, 3, :],
        ], dim=1)

        x = self.fusion(x)

        logits = self.head(x)

        return logits


def build_default_o4ffusiontransformer(freeze_baseline=True, device='cpu'):
    clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
    tokenizer = open_clip.get_tokenizer('ViT-32-B')
    clip_model.eval().to(device)

    if freeze_baseline:
        for param in clip_model.parameters():
            param.requires_grad = False

    model = O4FFusionTransformer(clip_model.encode_text, clip_model.encode_image, tokenizer)
    return model

In [ ]:
generator = torch.Generator().manual_seed(RANDOM_SEED)

train_dataloader = DataLoader(train_dataset, shuffle=True, generator=generator, batch_size=hps['batch_size'], num_workers=True)
test_dataloader = DataLoader(test_dataset, batch_size=hps['batch_size'], num_workers=True)

In [ ]:
model = build_default_o4ffusiontransformer(device=DEVICE)
model.to(DEVICE)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=hps['lr'])

In [ ]:
sum([param.numel() for param in model.parameters()])

1656024

In [ ]:
n_epochs = hps['n_epochs']
experiment_number = hps['experiment_number']
all_losses, losses_per_epoch = [], []

for epoch in range(1, n_epochs + 1):
    loss_accum = 0

    for batch_number, (frames, sentences, label) in enumerate(train_dataloader, start=1):
        optimizer.zero_grad()

        frames = frames.to(DEVICE)
        label = label.to(DEVICE)

        logits = model(frames, sentences)

        loss = loss_fn(logits, label)

        loss_accum += loss.item()
        all_losses.append(loss.item())

        loss.backward()

        optimizer.step()
        print(f'Batch #: [{batch_number}/{len(train_dataloader)}] Loss: [{loss.item()}]')

    loss_per_epoch = loss_accum / len(train_dataloader)
    losses_per_epoch.append(loss_per_epoch)

    torch.save(model.state_dict(), CKPT_PATH / f'o4f-fusion-transformer_{experiment_number}_e{epoch}.pt')

    print(f'Epoch: [{epoch}/{n_epochs}] Loss: [{loss_per_epoch}]')

Batch #: [1/298] Loss: [3.2841639518737793]
Batch #: [2/298] Loss: [3.8662164211273193]
Batch #: [3/298] Loss: [4.008325576782227]
Batch #: [4/298] Loss: [3.5606722831726074]
Batch #: [5/298] Loss: [3.6363301277160645]
Batch #: [6/298] Loss: [3.6167938709259033]
Batch #: [7/298] Loss: [3.17635440826416]
Batch #: [8/298] Loss: [3.314446449279785]
Batch #: [9/298] Loss: [3.5024325847625732]
Batch #: [10/298] Loss: [3.3782405853271484]
Batch #: [11/298] Loss: [3.242133378982544]
Batch #: [12/298] Loss: [3.5117905139923096]
Batch #: [13/298] Loss: [3.1853671073913574]
Batch #: [14/298] Loss: [3.226945400238037]
Batch #: [15/298] Loss: [3.068655252456665]
Batch #: [16/298] Loss: [3.409396171569824]
Batch #: [17/298] Loss: [2.970191717147827]
Batch #: [18/298] Loss: [3.277055263519287]
Batch #: [19/298] Loss: [3.3122897148132324]
Batch #: [20/298] Loss: [3.448302745819092]
Batch #: [21/298] Loss: [3.334914207458496]
Batch #: [22/298] Loss: [3.322258472442627]
Batch #: [23/298] Loss: [2.95886

RuntimeError: Parent directory /content/drive/MyDrive/Projects/SnuAiChallenge/Data/ckpt does not exist.